# TiRex-2 — DIMER forecasting tutorial

**Profile:** `TASK-INFERENCE`  
**Capability:** zero-shot probabilistic time-series forecasting with chronological evaluation

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. The default sample is demonstration evidence, not a production-quality or benchmark claim.

**Learning objectives:** resolve the immutable upstream model revision, validate a public/default input, run the supported task, inspect task-appropriate outputs, exercise an optional BYOD path, and export machine-readable outputs plus provenance.


## Prerequisites

Run in a fresh supported runtime. Install dependencies before importing PyTorch or Transformers. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

The installation cell installs this repository and its exact model-facing dependency versions. If installation replaces a pre-imported core framework, restart the runtime before continuing.

In [ ]:
%pip install -q -e .
import platform, numpy as np, pandas as pd, torch
print({'python':platform.python_version(),'torch':torch.__version__,'device':'cpu'})

## 2. Create a deterministic sample or load BYOD

The default sample is a deterministic synthetic seasonal/trend series. BYOD expects a CSV with `timestamp` and one or more numeric target columns; it is optional and gated off by default.

In [ ]:
USE_BYOD=False
if USE_BYOD:
    from google.colab import files
    name=next(iter(files.upload())); df=pd.read_csv(name); assert 'timestamp' in df.columns; values=df.drop(columns=['timestamp']).to_numpy(float).T
else:
    rng=np.random.default_rng(7); t=np.arange(256); values=(0.02*t+np.sin(t/8)+rng.normal(0,0.05,len(t)))[None,:]
print({'shape':values.shape,'sample':'BYOD' if USE_BYOD else 'deterministic synthetic'})

## 3. Chronological holdout and baseline

The final horizon is held out from the model context; no future target value is passed to the model. Last-value is the naive history-only baseline.

In [ ]:
from tirex_forecasting_pipeline import TiRexForecastPipeline, last_value_baseline, mae, rmse, MODEL_ID, MODEL_REVISION
horizon=32; context=values[:,:-horizon]; truth=values[:,-horizon:]; baseline=last_value_baseline(context,horizon)
print({'model_id':MODEL_ID,'revision':MODEL_REVISION,'context':context.shape[1],'horizon':horizon,'baseline_mae':mae(truth,baseline),'baseline_rmse':rmse(truth,baseline)})

## 4. Resolve and run TiRex-2

The loader passes the immutable model revision to the upstream TiRex-2 Hugging Face resolver. Output q=0.5 is treated as the median point forecast; all nine model quantiles remain available.

In [ ]:
pipe=TiRexForecastPipeline.from_pretrained(device='cpu')
result=pipe.forecast(context,horizon=horizon)
pred=result['median']
metrics={'mae':mae(truth,pred),'rmse':rmse(truth,pred),'baseline_mae':mae(truth,baseline),'baseline_rmse':rmse(truth,baseline)}
print(metrics)

## 5. Export forecasts and provenance

The CSV keeps time-step, variate, median, truth, and baseline aligned. JSON records model identity, runtime, horizon, context, and tutorial metrics.

In [ ]:
import json, os
os.makedirs('outputs',exist_ok=True)
rows=[]
for v in range(pred.shape[0]):
    for h in range(horizon): rows.append({'variate':v,'step':h+1,'median':float(pred[v,h]),'truth':float(truth[v,h]),'last_value_baseline':float(baseline[v,h])})
pd.DataFrame(rows).to_csv('outputs/tirex_forecast.csv',index=False)
prov={'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'metrics':metrics,'context_length':context.shape[1],'horizon':horizon,'quantile_levels':list(result['quantile_levels']),'runtime':{'python':platform.python_version(),'torch':torch.__version__,'device':'cpu'}}
with open('outputs/tirex_provenance.json','w') as f: json.dump(prov,f,indent=2)
print(['outputs/tirex_forecast.csv','outputs/tirex_provenance.json'])

## Interpretation and limits

The forecast is zero-shot; no gradient training or fine-tuning occurs. q=0.5 is a model median, and the other quantiles are model quantiles rather than guaranteed confidence intervals. Reported MAE/RMSE come from one chronological tutorial holdout and must be repeated over representative periods before deployment.

Successful execution proves that this repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Upstream model: https://huggingface.co/NX-AI/TiRex-2
- Upstream code: https://github.com/NX-AI/tirex-2
- Paper: https://arxiv.org/abs/2607.01204
